# Notebook 03: AI Refactoring

## Refactoring Messy Code with AI Assistance

This notebook covers practical refactoring patterns using AI. You'll see before/after examples, learn prompting strategies, and understand how to verify refactored code.

## 1. What is Refactoring?

Refactoring is improving code structure without changing its behavior.

**Goals:**
- Improve readability
- Reduce complexity
- Remove duplication
- Increase testability
- Follow SOLID principles

**Rule:** Always have tests before refactoring. If you don't have tests, write them first.

## 2. The Refactoring Workflow with AI

```
1. Identify target code
2. Describe the goal ("make it testable", "reduce complexity")
3. Provide constraints ("don't change the public API")
4. Generate refactored version
5. Verify behavior is preserved (tests)
6. Review changes
```

### Key Principle

**Never refactor without tests.** If you can't verify the behavior is preserved, don't refactor.

## 3. Refactoring Patterns

### Pattern 1: Extract Function

Break long functions into smaller, focused functions.

In [ ]:
# BEFORE: Long function doing everything
def process_order(order_data):
    # Validate input
    if not order_data:
        raise ValueError("Order data is empty")
    if 'items' not in order_data:
        raise ValueError("No items in order")
    if len(order_data['items']) == 0:
        raise ValueError("Order has no items")
    
    # Calculate subtotal
    subtotal = 0
    for item in order_data['items']:
        if 'price' not in item or 'qty' not in item:
            raise ValueError("Item missing price or qty")
        if item['price'] < 0:
            raise ValueError("Price cannot be negative")
        if item['qty'] <= 0:
            raise ValueError("Quantity must be positive")
        subtotal += item['price'] * item['qty']
    
    # Apply discount
    discount = 0
    if 'coupon' in order_data:
        coupon = order_data['coupon']
        if coupon == 'SAVE10':
            discount = subtotal * 0.10
        elif coupon == 'SAVE20':
            discount = subtotal * 0.20
        elif coupon == 'HALF':
            discount = subtotal * 0.50
        else:
            raise ValueError(f"Invalid coupon: {coupon}")
    
    # Calculate tax
    tax = (subtotal - discount) * 0.08
    
    # Calculate total
    total = subtotal - discount + tax
    
    return {
        'subtotal': subtotal,
        'discount': discount,
        'tax': tax,
        'total': total
    }

print("Before: 40+ lines in one function")

In [ ]:
# AFTER: Refactored into small, focused functions
from typing import Any

def validate_order_items(items: list[dict]) -> None:
    """Validate order items have required fields."""
    if not items:
        raise ValueError("Order has no items")
    
    for i, item in enumerate(items):
        if 'price' not in item or 'qty' not in item:
            raise ValueError(f"Item {i} missing price or qty")
        if item['price'] < 0:
            raise ValueError(f"Item {i} price cannot be negative")
        if item['qty'] <= 0:
            raise ValueError(f"Item {i} quantity must be positive")

def calculate_subtotal(items: list[dict]) -> float:
    """Calculate subtotal from items."""
    return sum(item['price'] * item['qty'] for item in items)

def apply_discount(subtotal: float, coupon: str) -> float:
    """Apply coupon discount."""
    discount_rates = {'SAVE10': 0.10, 'SAVE20': 0.20, 'HALF': 0.50}
    
    if coupon not in discount_rates:
        raise ValueError(f"Invalid coupon: {coupon}")
    
    return subtotal * discount_rates[coupon]

def calculate_tax(amount: float, rate: float = 0.08) -> float:
    """Calculate tax on amount."""
    return amount * rate

def process_order(order_data: dict[str, Any]) -> dict:
    """Process an order and return totals."""
    if not order_data or 'items' not in order_data:
        raise ValueError("Invalid order data")
    
    items = order_data['items']
    validate_order_items(items)
    
    subtotal = calculate_subtotal(items)
    discount = apply_discount(subtotal, order_data.get('coupon', ''))
    tax = calculate_tax(subtotal - discount)
    total = subtotal - discount + tax
    
    return {
        'subtotal': subtotal,
        'discount': discount,
        'tax': tax,
        'total': total
    }

print("After: Each function does one thing (Single Responsibility)")

### Pattern 2: Replace Conditionals with Polymorphism

When you have long if/elif chains, consider using classes or dictionaries.

In [ ]:
# BEFORE: Long if/elif chain
def calculate_shipping(method: str, weight: float, distance: float) -> float:
    if method == 'standard':
        return weight * 0.5 + distance * 0.1
    elif method == 'express':
        return weight * 1.0 + distance * 0.2 + 5.0
    elif method == 'overnight':
        return weight * 2.0 + distance * 0.5 + 15.0
    elif method == 'pickup':
        return 0.0
    else:
        raise ValueError(f"Unknown method: {method}")

print("Before: Adding a new method requires modifying this function")

In [ ]:
# AFTER: Strategy pattern with classes
from abc import ABC, abstractmethod

class ShippingStrategy(ABC):
    @abstractmethod
    def calculate(self, weight: float, distance: float) -> float:
        pass

class StandardShipping(ShippingStrategy):
    def calculate(self, weight: float, distance: float) -> float:
        return weight * 0.5 + distance * 0.1

class ExpressShipping(ShippingStrategy):
    def calculate(self, weight: float, distance: float) -> float:
        return weight * 1.0 + distance * 0.2 + 5.0

class OvernightShipping(ShippingStrategy):
    def calculate(self, weight: float, distance: float) -> float:
        return weight * 2.0 + distance * 0.5 + 15.0

class PickupShipping(ShippingStrategy):
    def calculate(self, weight: float, distance: float) -> float:
        return 0.0

# Registry makes adding new methods easy
SHIPPING_STRATEGIES = {
    'standard': StandardShipping(),
    'express': ExpressShipping(),
    'overnight': OvernightShipping(),
    'pickup': PickupShipping(),
}

def calculate_shipping(method: str, weight: float, distance: float) -> float:
    """Calculate shipping cost using strategy pattern."""
    if method not in SHIPPING_STRATEGIES:
        raise ValueError(f"Unknown method: {method}")
    return SHIPPING_STRATEGIES[method].calculate(weight, distance)

print("After: Adding new method = add new class + register it")

### Pattern 3: Extract Data Class

Replace dictionaries with typed data classes.

In [ ]:
# BEFORE: Using raw dictionaries
def create_user(data):
    return {
        'name': data['name'],
        'email': data['email'],
        'age': data.get('age'),
        'is_active': True,
    }

def send_welcome(user):
    print(f"Welcome {user['name']}! Your email is {user['email']}")

print("Before: No type safety, easy to misspell keys")

In [ ]:
# AFTER: Using Pydantic or dataclasses
from dataclasses import dataclass, field
from typing import Optional
from datetime import datetime

@dataclass
class User:
    name: str
    email: str
    age: Optional[int] = None
    is_active: bool = True
    created_at: datetime = field(default_factory=datetime.now)

def create_user(name: str, email: str, age: Optional[int] = None) -> User:
    """Create a new user with validation."""
    if not name or not name.strip():
        raise ValueError("Name cannot be empty")
    if '@' not in email:
        raise ValueError("Invalid email format")
    if age is not None and (age < 0 or age > 150):
        raise ValueError("Invalid age")
    
    return User(name=name.strip(), email=email.lower(), age=age)

def send_welcome(user: User) -> None:
    """Send welcome message to user."""
    print(f"Welcome {user.name}! Your email is {user.email}")

# Usage
user = create_user("Alice", "alice@example.com", 30)
send_welcome(user)
print(f"\nAfter: Type hints, validation, IDE autocomplete")

### Pattern 4: Replace Nested Conditionals with Guard Clauses

Use early returns to flatten nested if/else.

In [ ]:
# BEFORE: Deeply nested conditionals
def process_payment(order, user, payment_method):
    if user is not None:
        if user.is_active:
            if order is not None:
                if order.items:
                    if payment_method == 'credit':
                        # process credit card
                        return "Credit card processed"
                    elif payment_method == 'paypal':
                        # process paypal
                        return "PayPal processed"
                    else:
                        return "Invalid payment method"
                else:
                    return "Order has no items"
            else:
                return "No order provided"
        else:
            return "User is inactive"
    else:
        return "No user provided"

print("Before: 5 levels of nesting, hard to read")

In [ ]:
# AFTER: Guard clauses with early returns
from typing import Optional

def process_payment(order, user, payment_method: str) -> str:
    """Process payment for an order."""
    # Guard clauses
    if user is None:
        return "No user provided"
    if not user.is_active:
        return "User is inactive"
    if order is None:
        return "No order provided"
    if not order.items:
        return "Order has no items"
    
    # Main logic
    if payment_method == 'credit':
        return "Credit card processed"
    elif payment_method == 'paypal':
        return "PayPal processed"
    else:
        return "Invalid payment method"

print("After: Flat structure, easy to read")

### Pattern 5: Replace Stringly-Typed Code with Enums

Use enums instead of magic strings.

In [ ]:
# BEFORE: Magic strings
def get_status_color(status: str) -> str:
    if status == 'pending':
        return 'yellow'
    elif status == 'active':
        return 'green'
    elif status == 'cancelled':
        return 'red'
    else:
        return 'gray'

# Easy to typo: get_status_color('pendign')

print("Before: No type safety, typos silently fail")

In [ ]:
# AFTER: Enums with type safety
from enum import Enum

class OrderStatus(Enum):
    PENDING = 'pending'
    ACTIVE = 'active'
    CANCELLED = 'cancelled'

STATUS_COLORS = {
    OrderStatus.PENDING: 'yellow',
    OrderStatus.ACTIVE: 'green',
    OrderStatus.CANCELLED: 'red',
}

def get_status_color(status: OrderStatus) -> str:
    """Get color for order status."""
    return STATUS_COLORS.get(status, 'gray')

# Now typos are caught at runtime:
print(get_status_color(OrderStatus.PENDING))  # yellow
print(get_status_color(OrderStatus.ACTIVE))   # green

# This would fail with TypeError:
# get_status_color('pending')  # TypeError!

print("\nAfter: Type safety, IDE autocomplete, clear intent")

## 4. Testing Refactored Code

### Before Refactoring

1. **Write tests for current behavior** — Capture what the code does
2. **Run tests** — Ensure they pass with current code
3. **Commit** — Save working state

### After Refactoring

1. **Run same tests** — They should still pass
2. **Check coverage** — Ensure no lines were lost
3. **Review** — Code should be cleaner

In [ ]:
# Example: Test before and after refactoring
import pytest

# Original function
def apply_discount_original(price: float, discount_type: str) -> float:
    if discount_type == 'percentage':
        return price * 0.9
    elif discount_type == 'fixed':
        return price - 10
    elif discount_type == 'bogo':
        return price / 2
    else:
        return price

# Refactored function
from enum import Enum

class DiscountType(Enum):
    PERCENTAGE = 'percentage'
    FIXED = 'fixed'
    BOGO = 'bogo'

def apply_discount_refactored(price: float, discount_type: DiscountType) -> float:
    """Apply discount to price."""
    if discount_type == DiscountType.PERCENTAGE:
        return price * 0.9
    elif discount_type == DiscountType.FIXED:
        return price - 10
    elif discount_type == DiscountType.BOGO:
        return price / 2
    return price

# Tests (same for both versions)
def test_percentage_discount():
    assert apply_discount_original(100, 'percentage') == 90
    assert apply_discount_refactored(100, DiscountType.PERCENTAGE) == 90

def test_fixed_discount():
    assert apply_discount_original(100, 'fixed') == 90
    assert apply_discount_refactored(100, DiscountType.FIXED) == 90

def test_bogo_discount():
    assert apply_discount_original(100, 'bogo') == 50
    assert apply_discount_refactored(100, DiscountType.BOGO) == 50

# Run tests
test_percentage_discount()
test_fixed_discount()
test_bogo_discount()

print("All tests pass for both versions")
print("Refactored version adds type safety with same behavior")

## 5. Code Quality Metrics

### Before/After Measurement

Use these tools to measure improvement:

```bash
# Cyclomatic complexity
pip install radon
radon cc mymodule.py  # Lower is better (target < 10)

# Maintainability index
radon mi mymodule.py  # Higher is better (target > 20)

# Lines of code
radon raw mymodule.py

# Type coverage
pip install mypy
mypy mymodule.py

# Test coverage
pip install pytest-cov
pytest --cov=mymodule
```

In [ ]:
# Example metrics comparison

metrics = {
    "Metric": ["Cyclomatic Complexity", "Maintainability Index", 
              "Max Nesting Depth", "Type Coverage", "Test Coverage"],
    "Before": [15, 25, 5, 0, 40],
    "After": [6, 65, 2, 95, 85],
    "Target": ["< 10", "> 20", "< 3", "> 90%", "> 80%"]
}

print("\nCode Quality Metrics:")
print(f"{'Metric':<25} {'Before':<10} {'After':<10} {'Target':<15}")
print("-" * 60)
for i, metric in enumerate(metrics['Metric']):
    print(f"{metric:<25} {metrics['Before'][i]:<10} {metrics['After'][i]:<10} {metrics['Target'][i]:<15}")

## 6. Refactoring Prompt Templates

### Extract Function
```
Extract the [specific logic] in [function] into a separate function.
Keep the same behavior. Add type hints and docstrings.
```

### Simplify Conditional
```
The nested ifs in [function] are hard to read.
Refactor using early returns or a strategy pattern.
```

### Add Type Hints
```
Add comprehensive type hints to [module/file].
Include Pydantic models for complex structures.
```

### Make Async
```
Convert these synchronous database calls to async using SQLAlchemy 2.0 async.
Keep the same interface but use await.
```

### Replace with Data Class
```
Replace the dictionary usage in [function] with a Pydantic model.
Keep the same fields but add validation.
```

## 7. Key Takeaways

### Refactoring Checklist

- [ ] Write tests first (capture current behavior)
- [ ] Commit working code
- [ ] Describe the goal clearly to AI
- [ ] Provide constraints ("don't change public API")
- [ ] Generate refactored version
- [ ] Run tests — they must pass
- [ ] Measure metrics before/after
- [ ] Review for readability

### When to Refactor

- Code is duplicated
- Functions are too long (>30 lines)
- Nesting depth > 3
- Hard to write tests
- Adding features feels risky

### When NOT to Refactor

- No tests and can't add them
- Code is being deleted soon
- Deadline is imminent
- You don't understand the code well enough